# Aufbereitung Bevölkerungsdaten

Quelle: BFS, *Ständige und nichtständige Wohnbevölkerung nach Jahr, Kanton, Bevölkerungstyp, Staatsangehörigkeit, Geburtsstaat und Geschlecht* (px-x-0103010000_499), 2010 bis 2025.


**Was behalten wird:** alles, also alle Jahre, Kantone, Bevölkerungstypen, Staatsangehörigkeiten, Geburtsstaaten und Geschlechter inklusive aller Kombinationen. Weggelassen werden nur Zellen mit dem Wert 0 
**Wichtig beim Arbeiten mit den Daten:** Bei `pivot_table` immer `fill_value=0` setzen, damit fehlende Kombinationen als 0 statt als NA erscheinen.


In [1]:
import re
import numpy as np
import pandas as pd

## 1. PX-Datei einlesen

In [ ]:
text = open("Bevolkerungsdaten.px", encoding="utf-8-sig").read()
kopf, daten = text.split("DATA=", 1)
del text

# Dimensionen (Reihenfolge der Daten) und ihre Ausprägungen aus dem Kopf lesen
liste = lambda s: re.findall(r'"([^"]*)"', s)
dims = liste(re.search(r"STUB=(.*?);", kopf, re.S).group(1)) + liste(re.search(r"HEADING=(.*?);", kopf, re.S).group(1))
werte = {d: liste(re.search(r'VALUES\("%s"\)=(.*?);' % re.escape(d), kopf, re.S).group(1)) for d in dims}
{d: len(v) for d, v in werte.items()}

{'Jahr': 16,
 'Kanton': 28,
 'Bevölkerungstyp': 2,
 'Staatsangehörigkeit': 201,
 'Geburtsstaat': 202,
 'Geschlecht': 3}

In [3]:
# Zahlen direkt als numpy-Array einlesen (fehlende Werte wie "..." würden zu NaN)
daten = re.sub(r'"[^"]*"', " nan ", daten).rstrip().rstrip(";")
arr = np.fromstring(daten, sep=" ")
del daten

arr = arr.reshape([len(werte[d]) for d in dims])
print(arr.shape, "Zellen:", arr.size, "| fehlende Werte:", np.isnan(arr).sum(), f"| Nullen: {(arr == 0).mean():.1%}")

(16, 28, 2, 201, 202, 3) Zellen: 109138176 | fehlende Werte: 0 | Nullen: 97.6%


## 2. Kategorie „Ausland“ ergänzen

Bei Staatsangehörigkeit und Geburtsstaat wird je eine Kategorie `"Ausland"` hinzugefügt (Total minus Schweiz). Weil das für alle Kombinationen gemacht wird, gibt es z. B. auch:

- Staatsangehörigkeit Ausland + Geburtsstaat Schweiz: Ausländer, die in der Schweiz geboren sind (zweite Generation)
- Staatsangehörigkeit Schweiz + Geburtsstaat Ausland: eingebürgerte bzw. im Ausland geborene Schweizer
- Staatsangehörigkeit Ausland + Geburtsstaat Ausland: zugewanderte Ausländer

In [4]:
for d in ["Staatsangehörigkeit", "Geburtsstaat"]:
    achse = dims.index(d)
    total = arr.take([werte[d].index(f"{d} - Total")], axis=achse)
    schweiz = arr.take([werte[d].index("Schweiz")], axis=achse)
    arr = np.concatenate([arr, total - schweiz], axis=achse)
    werte[d] = werte[d] + ["Ausland"]

print(arr.shape, "| kleinster Wert (darf nicht negativ sein):", arr.min())

(16, 28, 2, 202, 203, 3) | kleinster Wert (darf nicht negativ sein): 0.0


## 3. In eine Tabelle umwandeln (ohne Nullen)

In [5]:
nz = np.nonzero(arr)  # Positionen aller Zellen ungleich 0, je ein Index-Array pro Dimension

bev = pd.DataFrame({d: pd.Categorical.from_codes(nz[k], werte[d]) for k, d in enumerate(dims)})
bev["Jahr"] = np.array(werte["Jahr"], dtype=int)[nz[0]]
bev["DATA"] = arr[nz]
del arr, nz

print(bev.shape)
bev.head(10)

(3166205, 7)


,Jahr,Kanton,Bevölkerungstyp,Staatsangehörigkeit,Geburtsstaat,Geschlecht,DATA
0,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Geburtsstaat - Total,Geschlecht - Total,7870134.0
1,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Geburtsstaat - Total,Mann,3877426.0
2,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Geburtsstaat - Total,Frau,3992708.0
3,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Schweiz,Geschlecht - Total,5794952.0
4,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Schweiz,Mann,2872045.0
5,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Schweiz,Frau,2922907.0
6,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Albanien,Geschlecht - Total,1723.0
7,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Albanien,Mann,756.0
8,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Albanien,Frau,967.0
9,2010,Schweiz,Ständige Wohnbevölkerung,Staatsangehörigkeit - Total,Andorra,Geschlecht - Total,43.0


## 4. Plausibilitätscheck

In [6]:
# Ständige Wohnbevölkerung Neuchâtel: Total, Schweiz, Ausland und Ausländeranteil
ne = bev[(bev["Kanton"] == "Neuchâtel") & (bev["Bevölkerungstyp"] == "Ständige Wohnbevölkerung")
         & (bev["Geschlecht"] == "Geschlecht - Total") & (bev["Geburtsstaat"] == "Geburtsstaat - Total")
         & bev["Staatsangehörigkeit"].isin(["Staatsangehörigkeit - Total", "Schweiz", "Ausland"])]
ne = ne.pivot_table(index="Jahr", columns="Staatsangehörigkeit", values="DATA", fill_value=0, observed=True)
ne["Ausländeranteil %"] = ne["Ausland"] / ne["Staatsangehörigkeit - Total"] * 100
ne.round(1)

Staatsangehörigkeit,Staatsangehörigkeit - Total,Schweiz,Ausland,Ausländeranteil %
Jahr,,,,
2010,172085.0,132499.0,39586.0,23.0
2011,173183.0,132685.0,40498.0,23.4
2012,174554.0,132652.0,41902.0,24.0
2013,176402.0,132428.0,43974.0,24.9
2014,177327.0,132575.0,44752.0,25.2
2015,178107.0,132556.0,45551.0,25.6
2016,178567.0,132878.0,45689.0,25.6
2017,177964.0,132811.0,45153.0,25.4
2018,176850.0,132143.0,44707.0,25.3


In [7]:
t = bev[(bev["Geschlecht"] == "Geschlecht - Total") & (bev["Geburtsstaat"] == "Geburtsstaat - Total")]

# Summe der Kantone muss dem Total Schweiz entsprechen
s = t.pivot_table(index=["Jahr", "Bevölkerungstyp", "Staatsangehörigkeit"], columns="Kanton", values="DATA", fill_value=0, observed=True)
print("Grösste Abweichung Kantone vs. Schweiz:", (s.drop(columns="Schweiz").sum(axis=1) - s["Schweiz"]).abs().max())

# Summe der einzelnen Länder muss dem Total entsprechen (ohne Total und ohne die neue Kategorie Ausland)
gruppen = ["Jahr", "Kanton", "Bevölkerungstyp"]
laender = t[~t["Staatsangehörigkeit"].isin(["Staatsangehörigkeit - Total", "Ausland"])].groupby(gruppen, observed=True)["DATA"].sum()
total = t[t["Staatsangehörigkeit"] == "Staatsangehörigkeit - Total"].groupby(gruppen, observed=True)["DATA"].sum()
print("Grösste Abweichung Länder vs. Total:", (laender - total).abs().max())

Grösste Abweichung Kantone vs. Schweiz: 0.0
Grösste Abweichung Länder vs. Total: 0.0


In [8]:
# Die 10 grössten ausländischen Gruppen in Neuchâtel 2016
t[(t["Kanton"] == "Neuchâtel") & (t["Jahr"] == 2016) & (t["Bevölkerungstyp"] == "Ständige Wohnbevölkerung")
  & ~t["Staatsangehörigkeit"].isin(["Staatsangehörigkeit - Total", "Schweiz", "Ausland"])].nlargest(10, "DATA")[["Staatsangehörigkeit", "DATA"]]

,Staatsangehörigkeit,DATA
1278001,Portugal,13766.0
1277226,Frankreich,7769.0
1277663,Italien,7197.0
1278182,Spanien,2587.0
1278543,Kosovo,1147.0
1278308,Türkiye,805.0
1279222,Eritrea,786.0
1277114,Deutschland,658.0
1278482,Bosnien und Herzegowina,593.0
1276998,Belgien,455.0


## 5. Als Parquet speichern

In [ ]:
bev.to_parquet("bevoelkerung.parquet", index=False)
#  einlesen mit: bev = pd.read_parquet("bevoelkerung.parquet")

## 6. Kriminalitätsdaten von CSV in Parquet umwandeln

in anderen Notebook dann `pd.read_parquet("beschuldigte.parquet")`. öfnnen

nur **einmal** laufen lassen

In [ ]:
#import os
#spalten = ["Straftat", "Kanton", "Aufenthaltsgruppe", "Geschlecht", "Altersklasse"]
#df_b = pd.read_csv("beschuldigte.csv", dtype={c: "category" for c in spalten})
#df_b.to_parquet("beschuldigte.parquet", index=False)

#print(df_b.shape)
#print(f"CSV:     {os.path.getsize('beschuldigte.csv') / 1e9:.2f} GB")
#print(f"Parquet: {os.path.getsize('beschuldigte.parquet') / 1e9:.2f} GB")

(34927605, 7)
CSV:     4.61 GB
Parquet: 0.01 GB
